# Demand & Available Cabs — Model comparison notebook

This notebook trains **LSTM**, **RandomForest**, **XGBoost**, **LightGBM**, and **CatBoost** on your dataset and compares their performance. It also selects the best model (by *accuracy* defined below) and uses it to predict future `demand` and `available_cabs`, then computes `shortage = demand - available_cabs`.

Path to data used in this notebook: `/mnt/data/Final_Processed.csv` (the file you provided).

### Notes before running
- Make sure you have GPU drivers and libraries installed (CUDA + cuDNN) and the following Python packages: `pandas, numpy, scikit-learn, tensorflow, xgboost, lightgbm, catboost, matplotlib, seaborn, joblib`.
- For XGBoost/LightGBM/CatBoost to use GPU, install GPU-enabled builds:
  - `xgboost` (gpu support) or `pip install xgboost` where GPU build is available.
  - `lightgbm` with `pip install lightgbm` built with GPU support (or build from source).
  - `catboost` supports GPU via `task_type='GPU'`.
- LSTM uses TensorFlow and will attempt to use available GPU.
- This notebook contains training cells — they will run on your machine when executed in VS Code. This file only *writes* the notebook; it does not run heavy training here.

## 1) Environment check and imports

In [5]:
# Check GPU availability and import libs
import os
print('Python executable:', os.sys.executable)

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

# GPU libraries (import where available)
try:
    import tensorflow as tf
    print('TensorFlow version:', tf.__version__)
    gpus = tf.config.list_physical_devices('GPU')
    print('TensorFlow GPUs:', gpus)
except Exception as e:
    print('TensorFlow import error or no GPU available:', e)

try:
    import xgboost as xgb
    print('XGBoost version:', xgb.__version__)
except Exception as e:
    print('XGBoost import problem:', e)

try:
    import lightgbm as lgb
    print('LightGBM version:', lgb.__version__)
except Exception as e:
    print('LightGBM import problem:', e)

try:
    from catboost import CatBoostRegressor
    import catboost
    print('CatBoost version:', catboost.__version__)
except Exception as e:
    print('CatBoost import problem:', e)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

print('\nEnvironment check complete. If you want GPU acceleration, run this notebook in an environment with CUDA/cuDNN and GPU-enabled builds of the libraries.')


Python executable: c:\Users\adity\AppData\Local\Programs\Python\Python311\python.exe
TensorFlow version: 2.19.0
TensorFlow GPUs: []
XGBoost version: 3.0.0
LightGBM version: 4.6.0
CatBoost version: 1.2.8

Environment check complete. If you want GPU acceleration, run this notebook in an environment with CUDA/cuDNN and GPU-enabled builds of the libraries.


## 2) Load data and basic preprocessing

In [6]:
# CELL: Load dataset
csv_path = r"C:\Users\adity\Downloads\MMDS\Project\Final_Processed.csv"

if not os.path.exists(csv_path):
    raise FileNotFoundError(
        "CSV not found at C:\\Users\\adity\\Downloads\\MMDS\\Project\\Final_Processed.csv. "
        "Please place Final_Processed.csv there."
    )

df = pd.read_csv(csv_path)
print("rows, cols =", df.shape)
df.head()


rows, cols = (347988, 14)


,ride_id,datetime,ride_status,pickup_area,drop_area,year,month,day,hour,date,time_zone,demand,available_cabs,shortage
0,BKG002834,2024-01-01T14:59:00.000Z,Success,Anekal,Bellandur,2024,1,1,14,2024-01-01,midday,16,24,-8
1,BKG004040,2024-01-01T12:47:00.000Z,Cancelled by Driver,Anekal,Yeshwanthpur,2024,1,1,12,2024-01-01,midday,16,24,-8
2,BKG034183,2024-01-01T14:38:00.000Z,Success,Anekal,RT Nagar,2024,1,1,14,2024-01-01,midday,16,24,-8
3,BKG042360,2024-01-01T13:34:00.000Z,Success,Anekal,Kengeri,2024,1,1,13,2024-01-01,midday,16,24,-8
4,BKG059450,2024-01-01T14:30:00.000Z,Cancelled by Driver,Anekal,Magadi Road,2024,1,1,14,2024-01-01,midday,16,24,-8


### Ensure columns exist and types
- Expected columns: `ride_id, datetime, ride_status, pickup_area, drop_area, year, month, day, hour, date, time_zone, demand, available_cabs, shortage`

In [7]:
expected = ['ride_id','datetime','ride_status','pickup_area','drop_area','year','month','day','hour','date','time_zone','demand','available_cabs','shortage']
for c in expected:
    if c not in df.columns:
        print('Missing expected column:', c)
print('Columns present:', df.columns.tolist())


Columns present: ['ride_id', 'datetime', 'ride_status', 'pickup_area', 'drop_area', 'year', 'month', 'day', 'hour', 'date', 'time_zone', 'demand', 'available_cabs', 'shortage']


## 3) Feature engineering
- We'll make time-related features and one-hot encode pickup_area (if needed). We'll train two separate models (one for `demand`, one for `available_cabs`) — or train a multi-output model. For simplicity, we'll train separate models and compare them.


In [8]:
df2 = df.copy()
# Ensure datetime
if 'datetime' in df2.columns:
    df2['datetime'] = pd.to_datetime(df2['datetime'])
else:
    df2['datetime'] = pd.to_datetime(df2['date'].astype(str) + ' ' + df2['hour'].astype(str)+':00')

df2['dayofweek'] = df2['datetime'].dt.dayofweek
df2['is_weekend'] = df2['dayofweek'].isin([5,6]).astype(int)

# If pickup_area is categorical, encode it
if df2['pickup_area'].dtype == 'object' or not np.issubdtype(df2['pickup_area'].dtype, np.number):
    df2 = pd.get_dummies(df2, columns=['pickup_area'], drop_first=True)

# Fill missing values in numeric targets if any
df2['demand'] = df2['demand'].astype(float)
df2['available_cabs'] = df2['available_cabs'].astype(float)
df2 = df2.sort_values('datetime').reset_index(drop=True)
display(df2.head())


,ride_id,datetime,ride_status,drop_area,year,month,day,hour,date,time_zone,...,pickup_area_Yeshwanthpur Track,pickup_area_Yeshwanthpur Trail,pickup_area_Yeshwanthpur Valley,pickup_area_Yeshwanthpur View,pickup_area_Yeshwanthpur Vista,pickup_area_Yeshwanthpur Way,pickup_area_Yeshwanthpur Woods,pickup_area_Yeshwanthpur Works,pickup_area_Yeshwanthpur Yard,pickup_area_Yeshwanthpur Zone
0,BKG012767,2024-01-01 00:00:00+00:00,Success,Koramangala,2024,1,1,0,2024-01-01,late_night,...,False,False,False,False,False,False,False,False,False,False
1,BKG164074,2024-01-01 00:00:00+00:00,Incomplete,Richmond Town,2024,1,1,0,2024-01-01,late_night,...,False,False,False,False,False,False,False,False,False,False
2,BKG121398,2024-01-01 00:00:00+00:00,Cancelled by Driver,Kammanahalli,2024,1,1,0,2024-01-01,late_night,...,False,False,False,False,False,False,False,False,False,False
3,BKG038517,2024-01-01 00:00:00+00:00,Success,Richmond Town,2024,1,1,0,2024-01-01,late_night,...,False,False,False,False,False,False,False,False,False,False
4,BKG002598,2024-01-01 00:00:00+00:00,Success,Cox Town,2024,1,1,0,2024-01-01,late_night,...,False,False,False,False,False,False,False,False,False,False


## 4) Train/test split
We will split by time (no leakage): last 20% as test.

In [9]:
test_frac = 0.2
n = len(df2)
train_df = df2.iloc[:int((1-test_frac)*n)].copy()
test_df  = df2.iloc[int((1-test_frac)*n):].copy()
print('Train rows:', len(train_df), 'Test rows:', len(test_df))


Train rows: 278390 Test rows: 69598


### Select features
Drop identifiers and target columns when building X. We'll auto-detect numeric features.

In [10]:
drop_cols = ['ride_id','datetime','date','ride_status','drop_area','shortage']
targets = ['demand','available_cabs']
features = [c for c in train_df.columns if c not in drop_cols + targets]
print('Using features:', features)


Using features: ['year', 'month', 'day', 'hour', 'time_zone', 'dayofweek', 'is_weekend', 'pickup_area_Adugodi 2nd', 'pickup_area_Adugodi 3rd', 'pickup_area_Adugodi 4th', 'pickup_area_Adugodi 5th', 'pickup_area_Adugodi 6th', 'pickup_area_Adugodi Alcove', 'pickup_area_Adugodi Alley', 'pickup_area_Adugodi Arc', 'pickup_area_Adugodi Arcade', 'pickup_area_Adugodi Area', 'pickup_area_Adugodi Bay', 'pickup_area_Adugodi Boulevard', 'pickup_area_Adugodi Bridge', 'pickup_area_Adugodi Circle', 'pickup_area_Adugodi Close', 'pickup_area_Adugodi Colony', 'pickup_area_Adugodi Commons', 'pickup_area_Adugodi Complex', 'pickup_area_Adugodi Court', 'pickup_area_Adugodi Cove', 'pickup_area_Adugodi Crescent', 'pickup_area_Adugodi Cross', 'pickup_area_Adugodi Cut', 'pickup_area_Adugodi Cutting', 'pickup_area_Adugodi Dam', 'pickup_area_Adugodi Depot', 'pickup_area_Adugodi District', 'pickup_area_Adugodi Drive', 'pickup_area_Adugodi Enclave', 'pickup_area_Adugodi Esplanade', 'pickup_area_Adugodi Estate', 'pic

## 5) Scaling
We'll use MinMaxScaler for LSTM and StandardScaler for tree models (but tree models don't strictly need scaling).

In [11]:
X_train = train_df[features].values
X_test = test_df[features].values
y_train = train_df[targets].values
y_test = test_df[targets].values

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)
y_train_scaled = scaler_y.fit_transform(y_train)
y_test_scaled = scaler_y.transform(y_test)

joblib.dump(scaler_X, 'scaler_X.joblib')
joblib.dump(scaler_y, 'scaler_y.joblib')
print('Scalers saved: scaler_X.joblib, scaler_y.joblib')


ValueError: could not convert string to float: 'late_night'

## 6) Prepare LSTM sequences (if using LSTM)
We'll create sequences of length `seq_len` (e.g., 24) using scaled X and y for training LSTM to predict both targets.

In [ ]:
def make_sequences(X, y, seq_len=24):
    Xs, ys = [], []
    for i in range(seq_len, len(X)):
        Xs.append(X[i-seq_len:i])
        ys.append(y[i])
    return np.array(Xs), np.array(ys)

seq_len = 24  # you can change (e.g., 24 for 24 hours)
X_train_seq, y_train_seq = make_sequences(X_train_scaled, y_train_scaled, seq_len)
X_test_seq, y_test_seq = make_sequences(X_test_scaled, y_test_scaled, seq_len)
print('LSTM shapes:', X_train_seq.shape, y_train_seq.shape)


## 7) Model training cells
Below are ready-to-run training cells for each model. They are **commented** to avoid accidental long runs. Remove comments to run. Each model will save a trained artifact.

In [ ]:
# -------------------------
# 7a) RandomForest
# -------------------------
rf_model = RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=42)
# To train, uncomment the next two lines:
# rf_model.fit(X_train, y_train)
# joblib.dump(rf_model, 'rf_model.joblib')

# -------------------------
# 7b) XGBoost (GPU if available)
# -------------------------
xgb_model = None
try:
    xgb_params = {
        'objective': 'reg:squarederror',
        'tree_method': 'gpu_hist',
        'predictor': 'gpu_predictor',
        'verbosity': 1,
        'seed': 42
    }
    xgb_model = xgb.XGBRegressor(**xgb_params, n_estimators=500)
    # To train: xgb_model.fit(X_train, y_train)
    # joblib.dump(xgb_model, 'xgb_model.joblib')
except Exception as e:
    print('XGBoost init problem (GPU maybe not available):', e)

# -------------------------
# 7c) LightGBM (GPU if available)
# -------------------------
lgb_model = None
try:
    lgb_params = {'device': 'gpu', 'verbosity': -1}
    # Using scikit-learn API wrapper
    lgb_model = lgb.LGBMRegressor(n_estimators=1000, **lgb_params)
    # To train: lgb_model.fit(X_train, y_train)
    # joblib.dump(lgb_model, 'lgb_model.joblib')
except Exception as e:
    print('LightGBM init problem:', e)

# -------------------------
# 7d) CatBoost (GPU if available)
# -------------------------
cat_model = None
try:
    cat_model = CatBoostRegressor(iterations=1000, learning_rate=0.1, depth=6, task_type='GPU', verbose=100)
    # To train: cat_model.fit(X_train, y_train)
    # cat_model.save_model('cat_model.cbm')
except Exception as e:
    print('CatBoost init problem:', e)

# -------------------------
# 7e) LSTM (TensorFlow) - multi-output
# -------------------------
lstm_model = None
try:
    tf.keras.backend.clear_session()
    inp_shape = (X_train_seq.shape[1], X_train_seq.shape[2]) if X_train_seq.size else (seq_len, X_train.shape[1])
    model = Sequential()
    model.add(LSTM(128, input_shape=inp_shape, return_sequences=False))
    model.add(Dropout(0.2))
    model.add(Dense(64, activation='relu'))
    model.add(Dense(len(targets)))
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    lstm_model = model
    # To train, uncomment the following:
    # es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    # chk = ModelCheckpoint('lstm_best.h5', save_best_only=True)
    # lstm_model.fit(X_train_seq, y_train_seq, validation_data=(X_test_seq, y_test_seq), epochs=100, batch_size=64, callbacks=[es,chk])
    # lstm_model.save('lstm_model.h5')
except Exception as e:
    print('LSTM build problem (maybe TF not installed or shapes mismatch):', e)

print('Model initialization complete. To train, run the training lines in each section (uncomment).')


## 8) Evaluation helper functions
We compute MAE, RMSE, R2. For *accuracy* (user requested), we define two accuracy types:
1. **within_10pct**: prediction considered correct if |pred - actual| <= 10% of actual (or 1 if actual is zero)
2. **within_1unit**: absolute error <= 1 (sensible for cab counts)

We'll compute these per-target and average to pick best model.


In [ ]:
def evaluate_regression(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    r2 = r2_score(y_true, y_pred)
    # within_10pct accuracy
    eps = np.maximum(0.1 * np.abs(y_true), 1.0)  # tolerance: 10% or 1
    within_10pct = np.mean(np.abs(y_pred - y_true) <= eps)
    within_1unit = np.mean(np.abs(y_pred - y_true) <= 1.0)
    return {'mae': mae, 'rmse': rmse, 'r2': r2, 'within_10pct': within_10pct, 'within_1unit': within_1unit}

def evaluate_and_format(y_true, y_pred):
    res = evaluate_regression(y_true, y_pred)
    return pd.Series(res)


## 9) Example evaluation cell (fill in with trained models)
After training models, load them and run the evaluation below. Replace model loading lines with your model paths.

In [ ]:
# Example: load trained models and evaluate (uncomment after you have trained and saved them)
results = {}

# RandomForest example (if trained and saved):
# rf_model = joblib.load('rf_model.joblib')
# preds_rf = rf_model.predict(X_test)
# results['rf'] = {}
# results['rf']['demand'] = evaluate_regression(y_test[:,0], preds_rf[:,0] if preds_rf.ndim>1 else preds_rf)
# results['rf']['available_cabs'] = evaluate_regression(y_test[:,1], preds_rf[:,1] if preds_rf.ndim>1 else preds_rf)

# For XGBoost / LightGBM / CatBoost, same pattern: load/predict/evaluate

print('Fill this cell with your model load & evaluate steps after training.')


## 10) Compare models into a table and select best model
We'll produce a DataFrame where rows are models and columns include metrics for both targets and the chosen accuracy measure (average of within_10pct for demand & available_cabs). We'll select the best by that accuracy.

In [ ]:
# Example structure for results DataFrame:
columns = ['model','demand_mae','demand_rmse','demand_r2','demand_within_10pct','available_mae','available_rmse','available_r2','available_within_10pct','avg_within_10pct']
results_df = pd.DataFrame(columns=columns)

# After filling results, choose best model by avg_within_10pct
# best_model_row = results_df.sort_values('avg_within_10pct', ascending=False).iloc[0]
# print('Best model by within_10pct accuracy:', best_model_row['model'])

print('This cell prepares a results DataFrame. Populate it with actual model evaluations.')


## 11) Predict future values given a date (dd mm yyyy) and compute shortage
We will implement a helper `predict_for_date(date_str)` that:
1. Builds input features for the requested date/time (you can decide hour). 
2. Uses the selected best model to predict demand and available_cabs.
3. Returns predicted demand, available_cabs and shortage = demand - available_cabs.

Note: To predict future values you must supply or simulate other features (pickup_area etc.). This function uses the last known row's categorical one-hot encoding and applies the date/time features.


In [ ]:
from datetime import datetime

def build_features_for_datetime(dt: datetime, template_row: pd.Series, features_list=features):
    # Create a feature vector based on template row and replace time features
    row = template_row.copy()
    row['year'] = dt.year
    row['month'] = dt.month
    row['day'] = dt.day
    row['hour'] = dt.hour
    row['dayofweek'] = dt.dayofweek
    row['is_weekend'] = int(dt.dayofweek in [5,6])
    # If pickup_area dummies exist, keep same distribution (uses last known)
    X = row[features_list].values.reshape(1, -1)
    Xs = scaler_X.transform(X)
    return Xs

def predict_for_date(date_str: str, time_hour: int=9, best_model=None, template_row=None):
    # date_str format: 'dd mm yyyy'
    dt = datetime.strptime(date_str.strip(), '%d %m %Y')
    dt = dt.replace(hour=time_hour)
    if best_model is None or template_row is None:
        raise ValueError('Please pass best_model and template_row (a recent row from the dataset)')
    Xs = build_features_for_datetime(dt, template_row, features)
    # If best_model is an sklearn-like regressor that predicts both targets, use directly
    ypred_scaled = best_model.predict(Xs)
    # If model predicts single target, adapt accordingly (not handled automatically here)
    ypred = scaler_y.inverse_transform(ypred_scaled.reshape(1,-1))
    demand_pred, avail_pred = float(ypred[0,0]), float(ypred[0,1])
    shortage = demand_pred - avail_pred
    return {'datetime': dt, 'demand': demand_pred, 'available_cabs': avail_pred, 'shortage': shortage}

print('Helper prediction functions created. Example usage:')
print("predict_for_date('25 12 2025', time_hour=9, best_model=my_model, template_row=df2.iloc[-1])")


## 12) How to run this notebook in VS Code with GPU (short checklist)
1. Install NVIDIA drivers and CUDA (compatible with your TensorFlow/XGBoost/LightGBM versions).
2. Create a conda environment and install packages, for example:
```
conda create -n mmds python=3.10 -y
conda activate mmds
pip install pandas numpy scikit-learn matplotlib seaborn joblib
pip install tensorflow  # or tensorflow-gpu depending on package availability
pip install xgboost lightgbm catboost
```
3. Open VS Code, install Python extension, and open this `.ipynb` file. Run cells.
4. Monitor GPU usage with `nvidia-smi`.

## 13) Final notes
- This notebook is intentionally modular: training lines are commented so you can control runs and resources.
- The notebook defines an *accuracy* as within-10% and within-1-unit measures because classic "accuracy" isn't defined for regression. We pick best model by average within_10pct across both targets (you can change the selection logic).
- After training, populate the evaluation cell to build the `results_df` and identify the best model. Then use `predict_for_date()` with that saved model.
